In [0]:
%sql
CREATE CATALOG if not EXISTS chemdata;
USE CATALOG chemdata;
create schema if not exists raw;
use schema raw;
create volume if not exists raw_data;

In [0]:
import hashlib
import requests
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
from ord_schema.proto import reaction_pb2
from ord_schema import message_helpers

In [0]:
DATASET_ID = "ord_dataset-00005539a1e04c809a9a78647bea649c"

SOURCE_URL = (
    "https://huggingface.co/datasets/"
    "open-reaction-database/ord-data/resolve/main/"
    "data/00/"
    f"{DATASET_ID}.parquet"
)

VOLUME_PATH = Path("/Volumes/chemdata/raw/raw_data")
DESTINATION = VOLUME_PATH / f"{DATASET_ID}.parquet"

In [0]:
def download_file(url: str, destination: Path):
    destination.parent.mkdir(parents=True, exist_ok=True)

    sha256 = hashlib.sha256()

    with requests.get(url, stream=True, timeout=300) as response:
        response.raise_for_status()

        with open(destination, "wb") as file:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    file.write(chunk)
                    sha256.update(chunk)

    return sha256.hexdigest()



In [0]:
download_file(
    SOURCE_URL,
    DESTINATION,
)

In [0]:
df = pd.read_parquet(str(DESTINATION))

In [0]:
df.head()

In [0]:
rxn = df.iloc[0]["reaction"]
rxn

In [0]:
def deserialize_reaction(reaction_bytes):
    if reaction_bytes is None:
        return None

    reaction = reaction_pb2.Reaction()
    reaction.ParseFromString(reaction_bytes)
    return reaction
    # return message_helpers.message_to_row(reaction)

In [0]:
df["deserialized_reaction"] = df["reaction"].apply(deserialize_reaction)
df.head()

In [0]:
reactions_df = message_helpers.messages_to_dataframe(df["deserialized_reaction"])
reactions_df.head()

In [0]:
reactions_df["outcomes[0].products[0].measurements[0].percentage.value"].hist()

In [0]:
reaction = reaction_pb2.Reaction()
reaction.ParseFromString(rxn)

In [0]:
reaction

In [0]:
deser_rxn=deserialize_reaction(rxn)

In [0]:
deser_rxn